# Exercise P5.1: sklearn Preprocessing Pipeline
### STAT 540 — Week 5


## Overview

In this exercise, you will build a complete sklearn preprocessing pipeline using ColumnTransformer and compare it to the R recipes approach.

## Task 1: Build a ColumnTransformer Pipeline

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Load data (use any tabular dataset with mixed types)
# Example: Ames housing from openml or a CSV
from sklearn.datasets import fetch_openml
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df = ames.frame

# Identify column types
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Remove target from feature lists
target = "SalePrice"
num_cols = [c for c in num_cols if c != target]

# Build pipelines
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Split FIRST, then fit
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit on training only
preprocessor.fit(X_train)
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Before: {X_train.shape[1]} columns")
print(f"After: {X_train_proc.shape[1]} columns")

Before: 80 columns
After: 287 columns


**Your turn:** Why did the column count increase after preprocessing?

> The column count increased because of the `OneHotEncoder` within the `cat_pipe` (categorical pipeline) part of the `ColumnTransformer`. Categorical features (like `MSZoning`, `Street`, etc.) are transformed into multiple new binary columns (one for each unique category). For example, if a 'Color' column has 3 unique values, it will be replaced by 3 new columns (e.g., 'Color_Red', 'Color_Green', 'Color_Blue'). Since the dataset has many categorical columns with numerous unique values, this process significantly expands the total number of features (columns).

## Task 2: Compare R and Python Pipelines

| Concept | R (recipes) | Python (sklearn) |
|---------|-------------|-----------------|
| Define pipeline | `recipe() |> step_...()` | `Pipeline([("name", transformer)])` |
| Fit on training | `prep(rec, training = train_data)` | `preprocessor.fit(X_train)` |
| Apply to new data | `bake(rec_prepped, new_data = test_data)` | `preprocessor.transform(X_test)` |
| Select column types | `all_numeric_predictors()`, `all_nominal_predictors()` | `df.select_dtypes(include=[...])`, then passed to `ColumnTransformer` |
| Handle mixed types | One recipe, steps applied selectively via role/type selectors (e.g., `all_numeric_predictors()` vs `all_nominal_predictors()`) | `ColumnTransformer` routes numeric columns to `num_pipe` and categorical columns to `cat_pipe` in parallel |

## Task 3: Demonstrate Data Leakage

Show what happens when you fit on ALL data vs training only:

In [3]:
# WRONG: fit on all data
preprocessor_wrong = ColumnTransformer([("num", num_pipe, num_cols[:3])])
X_all_proc = preprocessor_wrong.fit_transform(X)  # Leakage!

# RIGHT: fit on training only
preprocessor_right = ColumnTransformer([("num", num_pipe, num_cols[:3])])
preprocessor_right.fit(X_train)

# Compare means
print("Leaky scaler mean:", preprocessor_wrong.transformers_[0][1].named_steps["scaler"].mean_[:3])
print("Correct scaler mean:", preprocessor_right.transformers_[0][1].named_steps["scaler"].mean_[:3])

Leaky scaler mean: [730.5         56.89726027  69.86369863]
Correct scaler mean: [730.90496575  56.84931507  70.27996575]


**Your turn:** Are the means different? Why does this matter for model evaluation?

> Yes, the means are different. This indicates data. When we fit the scaler on the entire dataset (including the test set), the calculated mean and standard deviation are influenced by the test data. This means information from the test set has "leaked" into our training pipeline. This matters for model evaluation because the test set is meant to simulate completely new, unseen data. If our preprocessing has already seen the test data, the model's performance on that test set will be overly optimistic. It won't accurately reflect how the model will perform in the real world on truly unseen data. Preprocessors must always be fitted ONLY on the training data to ensure a realistic evaluation.

```bash
git add week05/exercises/P5.1*
git commit -m "Complete Exercise P5.1: sklearn preprocessing pipeline"
git push origin main
```